# 14_statistical_analysis_and_confidence_intervals

Computes bootstrap confidence intervals and paired model comparisons on the held-out test set.

In [1]:


from pathlib import Path
import os, json, shutil, zipfile, glob, warnings, math, random
from datetime import datetime, timezone
import numpy as np
import pandas as pd

_candidate_bases = [Path("/content"), Path("/mnt/data"), Path("/tmp")]
def _base_is_usable(p):
    try:
        if not (p.exists() and os.access(p, os.W_OK)):
            return False
        test_project = p / "project_thermography_equine"
        return (not test_project.exists()) or os.access(test_project, os.W_OK)
    except Exception:
        return False
_default_base = next((p for p in _candidate_bases if _base_is_usable(p)), Path("/tmp"))
BASE_DIR = Path(os.environ.get("THERMO_BASE_DIR", str(_default_base)))
PROJECT_NAME = "project_thermography_equine"
PROJECT_ROOT = BASE_DIR / PROJECT_NAME

DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
METADATA_DIR = DATA_ROOT / "metadata"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"
PROCESSED_DIR = DATA_ROOT / "processed"
CLEAN_IMAGE_DIR = PROCESSED_DIR / "clean_images"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
GRADCAM_DIR = OUTPUT_ROOT / "gradcam"
CASE_REVIEW_DIR = OUTPUT_ROOT / "case_review"

for p in [PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, SPLIT_DATA_DIR, METADATA_DIR, ANNOTATIONS_DIR, PROCESSED_DIR,
          CLEAN_IMAGE_DIR, OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, GRADCAM_DIR, CASE_REVIEW_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [BASE_DIR, Path("/mnt/data")]

def existing_roots():
    return [p for p in SEARCH_ROOTS if p.exists()]

def safe_copy(src, dst, overwrite=False):
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        if src.resolve() == dst.resolve():
            return dst
    except Exception:
        pass
    if overwrite or not dst.exists():
        shutil.copy2(src, dst)
    return dst

def find_first(patterns, roots=None):
    roots = roots or existing_roots()
    if isinstance(patterns, str):
        patterns = [patterns]
    candidates = []
    for root in roots:
        for pattern in patterns:
            candidates.extend(sorted(root.rglob(pattern)))
    candidates = [p for p in candidates if p.is_file() and ".ipynb_checkpoints" not in p.parts]
    return candidates[0] if candidates else None

def recover_file(target_name, patterns=None, subdirs=("config","reports","tables","figures","models")):
    """Find uploaded or previously generated file and copy to standard project locations."""
    patterns = patterns or [target_name]
    dst_primary = CONFIG_DIR / target_name
    if dst_primary.exists():
        return dst_primary
    src = find_first(patterns)
    if src is None:
        return None

    suffix = Path(target_name).suffix.lower()
    if suffix in [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".svg"]:
        dst_primary = FIGURES_DIR / target_name
    elif suffix in [".pt", ".pth", ".joblib", ".pkl"]:
        dst_primary = MODELS_DIR / target_name
    elif target_name.startswith("table_"):
        dst_primary = TABLES_DIR / target_name
    else:
        dst_primary = CONFIG_DIR / target_name
    safe_copy(src, dst_primary)

    if suffix in [".csv", ".json", ".txt"]:
        safe_copy(src, CONFIG_DIR / target_name)
        safe_copy(src, REPORTS_DIR / target_name)
        if target_name.startswith("table_") or "table" in target_name:
            safe_copy(src, TABLES_DIR / target_name)
    return dst_primary

def recover_known_outputs():
    mapping = {
        "classical_features.csv": ["classical_features.csv", "classical_features*.csv"],
        "classical_feature_metadata.json": ["classical_feature_metadata.json"],
        "classical_feature_summary.csv": ["classical_feature_summary.csv"],
        "classical_model_selection_results.csv": ["classical_model_selection_results.csv", "classical_model_selection_results*.csv"],
        "selected_classical_model.json": ["selected_classical_model.json"],
        "selected_classical_model.joblib": ["selected_classical_model.joblib", "selected_classical_model*.joblib"],
        "classical_baseline_metrics.csv": ["classical_baseline_metrics.csv"],
        "table_classical_baseline_metrics.csv": ["table_classical_baseline_metrics.csv"],
        "classical_baseline_predictions.csv": ["classical_baseline_predictions.csv"],
        "cnn_model_metrics.csv": ["cnn_model_metrics.csv"],
        "table_cnn_model_metrics.csv": ["table_cnn_model_metrics.csv"],
        "cnn_model_predictions.csv": ["cnn_model_predictions.csv"],
        "cnn_model_record.json": ["cnn_model_record.json"],
        "cnn_training_history.csv": ["cnn_training_history.csv"],
        "model_ablation_results.csv": ["model_ablation_results.csv"],
        "model_ablation_summary.csv": ["model_ablation_summary.csv"],
        "table_model_ablation_summary.csv": ["table_model_ablation_summary.csv"],
        "methods_classical_baseline_text.txt": ["methods_classical_baseline_text.txt"],
        "methods_ablation_robustness_text.txt": ["methods_ablation_robustness_text.txt"],
        "fig_classical_baseline_test_roc.png": ["fig_classical_baseline_test_roc.png"],
        "fig_cnn_test_roc.png": ["fig_cnn_test_roc.png"],
    }
    status = {}
    for target, patterns in mapping.items():
        status[target] = str(recover_file(target, patterns)) if recover_file(target, patterns) else None

    for src in []:
        pass
    for root in existing_roots():
        for pat in ["cnn_*best*.pt", "cnn_*.pt", "*.pth"]:
            for src in sorted(root.rglob(pat)):
                if src.is_file() and ".ipynb_checkpoints" not in src.parts:
                    safe_copy(src, MODELS_DIR / src.name)

    extracted = []
    for root in existing_roots():
        for z in sorted(root.glob("*.zip")):
            lname = z.name.lower()
            if any(tok in lname for tok in ["image", "clean", "preprocess", "processed", "dataset", "raw", "annotation", "hotspot", "gradcam"]):
                target = PROJECT_ROOT / "_uploaded_zip_extracts" / z.stem
                target.mkdir(parents=True, exist_ok=True)
                marker = target / ".extracted"
                if not marker.exists():
                    with zipfile.ZipFile(z, "r") as zr:
                        zr.extractall(target)
                    marker.write_text(datetime.now(timezone.utc).isoformat(), encoding="utf-8")
                extracted.append(str(target))
    status["extracted_zip_dirs"] = extracted
    return status

recovery_status = recover_known_outputs()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_DIR:", CONFIG_DIR)
print("Recovered known outputs:")
for k, v in recovery_status.items():
    if v:
        print(" -", k, "->", v)

def read_csv_required(path, required_columns=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required file missing: {path}")
    df = pd.read_csv(path)
    if required_columns:
        missing = [c for c in required_columns if c not in df.columns]
        if missing:
            raise KeyError(f"{path.name} is missing required columns: {missing}")
    return df

def standard_metric_col(df):
    df = df.copy()
    if "auc" not in df.columns and "roc_auc" in df.columns:
        df["auc"] = df["roc_auc"]
    if "roc_auc" not in df.columns and "auc" in df.columns:
        df["roc_auc"] = df["auc"]
    if "average_precision" not in df.columns and "pr_auc" in df.columns:
        df["average_precision"] = df["pr_auc"]
    return df

def assert_no_test_selection(record_or_df=None):
    if isinstance(record_or_df, dict):
        if record_or_df.get("test_set_used_for_model_selection") is True:
            raise AssertionError("Test set was used for model selection; this violates the locked analysis protocol.")
    return True


PROJECT_ROOT: /content/project_thermography_equine
CONFIG_DIR: /content/project_thermography_equine/outputs/config
Recovered known outputs:


In [4]:

from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

RNG = np.random.default_rng(42)
N_BOOT = int(os.environ.get("THERMO_BOOTSTRAP_N", "1000"))

base = read_csv_required(CONFIG_DIR / "classical_baseline_predictions.csv", ["split","label_binary","classical_probability_pathological","classical_predicted_label_binary"])
cnn = read_csv_required(CONFIG_DIR / "cnn_model_predictions.csv", ["split","label_binary","cnn_probability_pathological","cnn_predicted_label_binary"])

base_test = base[base["split"]=="test"].copy().reset_index(drop=True)
cnn_test = cnn[cnn["split"]=="test"].copy().reset_index(drop=True)

def metrics_from_predictions(y, prob, pred):
    y = np.asarray(y).astype(int); prob = np.asarray(prob).astype(float); pred = np.asarray(pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        "auc": roc_auc_score(y, prob) if len(np.unique(y)) == 2 else np.nan,
        "average_precision": average_precision_score(y, prob) if len(np.unique(y)) == 2 else np.nan,
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "f1": f1_score(y, pred, zero_division=0),
        "sensitivity": tp/(tp+fn) if (tp+fn) else np.nan,
        "specificity": tn/(tn+fp) if (tn+fp) else np.nan,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }

def bootstrap_ci(df, prob_col, pred_col, model_name):
    y = df["label_binary"].astype(int).to_numpy()
    p = df[prob_col].astype(float).to_numpy()
    pred = df[pred_col].astype(int).to_numpy()
    point = metrics_from_predictions(y,p,pred)
    rows = []
    n = len(df)
    samples = {k: [] for k in ["auc","average_precision","accuracy","balanced_accuracy","f1","sensitivity","specificity"]}
    for b in range(N_BOOT):
        idx = RNG.integers(0, n, size=n)
        if len(np.unique(y[idx])) < 2:
            continue
        m = metrics_from_predictions(y[idx], p[idx], pred[idx])
        for k in samples:
            samples[k].append(m[k])
    for k, vals in samples.items():
        arr = np.asarray(vals, dtype=float)
        rows.append({
            "model_name": model_name,
            "metric": k,
            "point_estimate": float(point[k]),
            "mean": float(np.nanmean(arr)) if len(arr) else np.nan,
            "ci_lower": float(np.nanpercentile(arr, 2.5)) if len(arr) else np.nan,
            "ci_upper": float(np.nanpercentile(arr, 97.5)) if len(arr) else np.nan,
            "n_bootstrap": int(len(arr)),
            "n_test": int(n),
        })
    return pd.DataFrame(rows), point

base_ci, base_point = bootstrap_ci(base_test, "classical_probability_pathological", "classical_predicted_label_binary", "classical_logistic_l2")
cnn_ci, cnn_point = bootstrap_ci(cnn_test, "cnn_probability_pathological", "cnn_predicted_label_binary", "cnn_resnet18")
final_statistics_table = pd.concat([base_ci, cnn_ci], ignore_index=True)
auc_ci_table = final_statistics_table[final_statistics_table["metric"]=="auc"].copy()

# Paired bootstrap AUC difference: CNN minus classical, requiring row alignment by horse/image where possible.
paired = cnn_test.merge(
    base_test[["horse_id","image_name","classical_probability_pathological"]],
    on=[c for c in ["horse_id","image_name"] if c in cnn_test.columns and c in base_test.columns],
    how="inner"
)
diffs = []
if len(paired) > 0:
    y = paired["label_binary"].astype(int).to_numpy()
    p_cnn = paired["cnn_probability_pathological"].astype(float).to_numpy()
    p_base = paired["classical_probability_pathological"].astype(float).to_numpy()
    n = len(paired)
    for b in range(N_BOOT):
        idx = RNG.integers(0, n, size=n)
        if len(np.unique(y[idx])) < 2:
            continue
        diffs.append(roc_auc_score(y[idx], p_cnn[idx]) - roc_auc_score(y[idx], p_base[idx]))
diffs = np.asarray(diffs, dtype=float)
point_diff = cnn_point["auc"] - base_point["auc"]
model_comparison_table = pd.DataFrame([{
    "comparison": "cnn_resnet18_minus_classical_logistic_l2",
    "metric": "auc_difference",
    "point_difference": float(point_diff),
    "mean_difference": float(np.nanmean(diffs)) if len(diffs) else np.nan,
    "ci_lower": float(np.nanpercentile(diffs, 2.5)) if len(diffs) else np.nan,
    "ci_upper": float(np.nanpercentile(diffs, 97.5)) if len(diffs) else np.nan,
    "n_bootstrap": int(len(diffs)),
    "interpretation": "negative values favor classical model; positive values favor CNN",
}])

for name, df in [
    ("final_statistics_table.csv", final_statistics_table),
    ("auc_ci_table.csv", auc_ci_table),
    ("model_comparison_table.csv", model_comparison_table),
]:
    df.to_csv(CONFIG_DIR / name, index=False)
    df.to_csv(REPORTS_DIR / name, index=False)
    df.to_csv(TABLES_DIR / name, index=False)

stat_config = pd.DataFrame([{"n_bootstrap_requested": N_BOOT, "random_seed": 42, "test_set_used_for_selection": False}])
stat_config.to_csv(CONFIG_DIR / "statistical_config_used.csv", index=False)

base_auc = base_point["auc"]; cnn_auc = cnn_point["auc"]
winner = "classical feature-based model" if base_auc >= cnn_auc else "CNN model"
stat_text = f"""Statistical analysis summary

Bootstrap 95% confidence intervals were computed on the locked held-out test set.
Classical logistic baseline test AUC: {base_auc:.4f}.
CNN ResNet18 test AUC: {cnn_auc:.4f}.
The paired bootstrap AUC difference was computed as CNN minus classical: {point_diff:.4f}.
Negative values favor the classical model; positive values favor the CNN.
The point estimate favored the {winner}.
The test set was not used for model or threshold selection.
""".strip()
(REPORTS_DIR / "statistical_analysis_summary.txt").write_text(stat_text, encoding="utf-8")
(CONFIG_DIR / "statistical_analysis_summary.txt").write_text(stat_text, encoding="utf-8")
display(final_statistics_table)
display(model_comparison_table)
print(stat_text)


,model_name,metric,point_estimate,mean,ci_lower,ci_upper,n_bootstrap,n_test
0,classical_logistic_l2,auc,0.892308,0.892079,0.774931,0.979072,1000,53
1,classical_logistic_l2,average_precision,0.773189,0.779793,0.543440,0.939527,1000,53
2,classical_logistic_l2,accuracy,0.433962,0.435679,0.301887,0.566038,1000,53
3,classical_logistic_l2,balanced_accuracy,0.625000,0.626532,0.564516,0.695122,1000,53
4,classical_logistic_l2,f1,0.464286,0.459726,0.297872,0.618220,1000,53
5,classical_logistic_l2,sensitivity,1.000000,1.000000,1.000000,1.000000,1000,53
6,classical_logistic_l2,specificity,0.250000,0.253065,0.129032,0.390244,1000,53
7,cnn_resnet18,auc,0.925000,0.922123,0.842835,0.982703,1000,53
8,cnn_resnet18,average_precision,0.798452,0.798034,0.557157,0.962367,1000,53
9,cnn_resnet18,accuracy,0.830189,0.828566,0.716981,0.924528,1000,53


,comparison,metric,point_difference,mean_difference,ci_lower,ci_upper,n_bootstrap,interpretation
0,cnn_resnet18_minus_classical_logistic_l2,auc_difference,0.032692,0.033341,-0.038618,0.11932,1000,negative values favor classical model; positiv...


Statistical analysis summary

Bootstrap 95% confidence intervals were computed on the locked held-out test set.
Classical logistic baseline test AUC: 0.8923.
CNN ResNet18 test AUC: 0.9250.
The paired bootstrap AUC difference was computed as CNN minus classical: 0.0327.
Negative values favor the classical model; positive values favor the CNN.
The point estimate favored the CNN model.
The test set was not used for model or threshold selection.


## Completion
This notebook writes standardized outputs to `outputs/config`, `outputs/reports`, `outputs/tables`, and/or `outputs/figures`.